# Experiment: AMDock Ligand Import Chemical Profiling

Objective:
- Measure the chemical and molecule-processing cost of the current ligand import pipeline without job scheduling or database writes.
- Separate file counting, entry parsing, per-molecule RDKit/materialization work, fragment analysis, selected-fragment reload, and full batch materialization.

Success criteria:
- Obtain stage-level timings over the same input subset.
- Compare the manual step-by-step loop against `materialize_import_batch(...)` on the same records.
- Leave the notebook ready to rerun on any SDF file by changing a small config cell.


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

import json
import shutil
import statistics
import sys
import tempfile
import time
from collections import Counter
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path('/home/mario/PycharmProjects/AMDockVS')
MOLSUITE_ROOT = PROJECT_ROOT.parent / 'MolSuite'
for candidate in (PROJECT_ROOT / 'src', MOLSUITE_ROOT / 'src'):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

from rdkit import Chem
from rdkit import RDLogger

from amdockvs.chemistry.state import molecule_state_metadata
from amdockvs.chemistry.filtering import evaluate_small_molecule_filter_values
from amdockvs.io.loaders import stream_import_payload_batches
from amdockvs.io.parsers.readers import count_import_records, iter_import_entries
from amdockvs.io.payloads import ImportBatchPayload
from amdockvs.io.transformers.materializers import (
    _build_row,
    _compute_small_molecule_filter_values,
    _project_root_from_storage_dir,
    _source_properties_from_mapping,
    build_import_graph_payload,
    materialize_import_batch,
)
from amdockvs.models.molecules import MoleculeType
from amdockvs.molecules.fragments import analyze_ligand_fragments, selected_fragment_from_metadata
from amdockvs.molecules.fragments import _sanitize_fragment, _component_role, _canonical_smiles, _formula, _has_carbon, _has_metal_atoms, _is_simple_ion
from amdockvs.core.paths import managed_paths_for_source

RDLogger.DisableLog('rdApp.*')


## Plan

- Use a configurable subset of a real SDF file.
- Measure two counters first:
  - plain-text SDF block count
  - `count_import_records(...)`
- Parse entries once and keep the subset fixed.
- Run a manual per-molecule loop that mirrors the current ligand algorithm.
- Run `materialize_import_batch(...)` and `build_import_graph_payload(...)` on the same subset.
- Compare totals and rates.


In [ ]:
# Configuration
INPUT_FILE = Path('/home/mario/DATA/data/mario/PycharmProjects/MSDD/mid_test.sdf')
MAX_RECORDS = 1000
BATCH_SIZE = 1000
PRIMARY_ROLE = 'ligand'
PRIMARY_CONTEXT = 'general'
MOLECULE_KIND = 'small_molecule'
KEEP_WORKSPACE = False

assert INPUT_FILE.exists(), INPUT_FILE

workspace_dir = Path(tempfile.mkdtemp(prefix='amdockvs-chemical-profile-')).resolve()
storage_dir = workspace_dir / 'storage'
storage_dir.mkdir(parents=True, exist_ok=True)
project_root = _project_root_from_storage_dir(storage_dir)

config = {
    'input_file': str(INPUT_FILE),
    'max_records': MAX_RECORDS,
    'batch_size': BATCH_SIZE,
    'workspace_dir': str(workspace_dir),
    'storage_dir': str(storage_dir),
    'project_root': str(project_root),
}
pprint(config)


In [ ]:
# Count and parse a fixed subset of entries

def count_sdf_blocks(file_path: Path) -> int:
    total = 0
    with file_path.open('r', encoding='utf-8', errors='ignore') as handle:
        for line in handle:
            if line.strip() == '$$$$':
                total += 1
    return total


t0 = time.perf_counter()
sdf_blocks = count_sdf_blocks(INPUT_FILE)
t1 = time.perf_counter()
importable_records = count_import_records(INPUT_FILE)
t2 = time.perf_counter()
input_format, entry_iter = iter_import_entries(kind='ligand', file_path=INPUT_FILE)
entries = []
for entry in entry_iter:
    entries.append(dict(entry))
    if len(entries) >= MAX_RECORDS:
        break
t3 = time.perf_counter()

entry_summary = {
    'input_format': input_format,
    'subset_entries': len(entries),
    'sdf_blocks': sdf_blocks,
    'importable_records': importable_records,
    'count_sdf_blocks_s': round(t1 - t0, 6),
    'count_import_records_s': round(t2 - t1, 6),
    'parse_subset_entries_s': round(t3 - t2, 6),
}
pprint(entry_summary)


In [ ]:
# Helpers for stage-level profiling

def make_batch_payload(entries_subset: list[dict]) -> dict:
    return ImportBatchPayload(
        kind='ligand',
        file_path=INPUT_FILE,
        storage_dir=storage_dir,
        input_format='sdf',
        primary_role=PRIMARY_ROLE,
        primary_context=PRIMARY_CONTEXT,
        molecule_kind=MOLECULE_KIND,
        prefilter=None,
        extra_data_patch={},
        binding_site_specs=[],
        entries=list(entries_subset),
    ).model_dump(mode='json')


def summarize_timings(stage_seconds: dict[str, float], *, molecules_kept: int, entries_seen: int) -> list[tuple[str, dict[str, float]]]:
    rows = []
    total = sum(stage_seconds.values())
    for name, seconds in sorted(stage_seconds.items(), key=lambda item: item[1], reverse=True):
        rows.append((
            name,
            {
                'seconds': round(seconds, 6),
                'pct_of_profiled': round((seconds / total) * 100.0, 2) if total > 0 else 0.0,
                'ms_per_entry_seen': round((seconds / max(1, entries_seen)) * 1000.0, 4),
                'ms_per_molecule_kept': round((seconds / max(1, molecules_kept)) * 1000.0, 4),
            },
        ))
    return rows


def cleanup_workspace(path: Path) -> None:
    if path.exists() and not KEEP_WORKSPACE:
        shutil.rmtree(path, ignore_errors=True)


In [ ]:
# Manual step-by-step chemical pipeline profiling

def profile_manual_ligand_pipeline(entries_subset: list[dict]) -> dict:
    stage_seconds: Counter[str] = Counter()
    counters: Counter[str] = Counter()
    rows: list[dict] = []

    pipeline_started = time.perf_counter()
    for entry in entries_subset:
        counters['entries_seen'] += 1
        mol_block = str(entry.get('mol_block') or '')
        source_index = int(entry.get('source_index') or 0)
        name = str(entry.get('name') or f'{INPUT_FILE.stem}_{source_index}')

        t0 = time.perf_counter()
        mol = Chem.MolFromMolBlock(mol_block, sanitize=True, removeHs=False)
        stage_seconds['mol_from_mol_block'] += time.perf_counter() - t0
        if mol is None:
            counters['mol_none'] += 1
            continue

        t0 = time.perf_counter()
        paths = managed_paths_for_source(
            storage_root=storage_dir,
            role=PRIMARY_ROLE,
            source_file=INPUT_FILE,
            source_index=source_index,
            original_suffix='.sdf',
            current_suffix='.sdf',
        )
        stage_seconds['managed_paths_for_source'] += time.perf_counter() - t0
        stored_path = paths['original_path']
        current_path = paths['current_path']

        t0 = time.perf_counter()
        stored_path.write_text(mol_block, encoding='utf-8')
        stage_seconds['write_original_mol_block'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        metadata = molecule_state_metadata(mol)
        stage_seconds['molecule_state_metadata'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        row = _build_row(
            project_root=project_root,
            source_file=INPUT_FILE,
            source_index=source_index,
            name=name,
            n_atoms=mol.GetNumAtoms(),
            input_format='sdf',
            stored_path=stored_path,
            current_path=current_path,
            metadata=metadata,
            molecule_kind=MOLECULE_KIND,
            primary_role=PRIMARY_ROLE,
            primary_context=PRIMARY_CONTEXT,
        )
        stage_seconds['_build_row'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        row['source_properties'] = _source_properties_from_mapping(entry.get('source_properties'))
        stage_seconds['_source_properties_from_mapping'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        fragment_info = analyze_ligand_fragments(
            mol=mol,
            storage_root=storage_dir,
            role=PRIMARY_ROLE,
            storage_key=str(paths['key']),
            project_root=project_root,
        )
        stage_seconds['analyze_ligand_fragments'] += time.perf_counter() - t0
        if fragment_info is None:
            counters['fragment_info_none'] += 1
            continue

        t0 = time.perf_counter()
        metadata_map = dict(row.get('extra_data') or {})
        metadata_map['fragmentation'] = fragment_info
        selected_fragment = selected_fragment_from_metadata(metadata_map)
        stage_seconds['selected_fragment_from_metadata'] += time.perf_counter() - t0
        if selected_fragment is None:
            counters['selected_fragment_none'] += 1
            continue

        t0 = time.perf_counter()
        selected_rel_path = str(selected_fragment.get('path') or '').strip()
        if selected_rel_path:
            selected_abs_path = (project_root / selected_rel_path).resolve()
            if selected_abs_path.exists():
                current_path.write_text(selected_abs_path.read_text(encoding='utf-8'), encoding='utf-8')
        stage_seconds['copy_selected_fragment_text'] += time.perf_counter() - t0

        properties = dict(selected_fragment.get('properties') or {})
        state = dict(selected_fragment.get('state') or {})
        row['extra_data'] = metadata_map
        row['current_path'] = str(current_path.relative_to(project_root))
        row['n_atoms'] = int(properties.get('n_atoms') or row.get('n_atoms') or 0)
        for key in (
            'mw', 'exact_mw', 'logp', 'hbd', 'hba', 'tpsa', 'rotatable_bonds',
            'fragment_count', 'ring_count', 'aromatic_ring_count', 'hetero_atom_count',
            'heavy_atom_count', 'formal_charge', 'fraction_csp3',
        ):
            row[key] = properties.get(key)
        row['pains_matches'] = list(properties.get('pains_matches') or [])
        row['ro5_violations'] = list(properties.get('ro5_violations') or [])
        row['has_3d'] = bool(state.get('has_3d', row.get('has_3d')))
        row['has_hs'] = bool(state.get('has_hs', row.get('has_hs')))
        row['conformer_count'] = int(state.get('conformer_count', row.get('conformer_count') or 0) or 0)
        row['metadata_json'] = json.dumps(metadata_map, ensure_ascii=True)

        t0 = time.perf_counter()
        selected_mol = Chem.MolFromMolFile(str((project_root / row['current_path']).resolve()), sanitize=True, removeHs=False)
        stage_seconds['selected_mol_from_file'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        selected_report = _compute_small_molecule_filter_values(selected_mol, molecule_kind=MOLECULE_KIND)
        stage_seconds['_compute_small_molecule_filter_values'] += time.perf_counter() - t0
        if selected_report is None:
            counters['selected_report_none'] += 1
            continue

        rows.append(row)
        counters['rows_kept'] += 1

    stage_seconds['manual_total_wall'] += time.perf_counter() - pipeline_started
    graph_t0 = time.perf_counter()
    graph_payload = build_import_graph_payload(rows)
    stage_seconds['build_import_graph_payload'] += time.perf_counter() - graph_t0

    return {
        'rows': rows,
        'graph_payload': graph_payload,
        'stage_seconds': dict(stage_seconds),
        'counters': dict(counters),
    }

manual_profile = profile_manual_ligand_pipeline(entries)
manual_summary = {
    'entries_seen': manual_profile['counters'].get('entries_seen', 0),
    'rows_kept': manual_profile['counters'].get('rows_kept', 0),
    'rows_dropped': manual_profile['counters'].get('entries_seen', 0) - manual_profile['counters'].get('rows_kept', 0),
    'molecule_source_properties_total': sum(len(row.get('source_properties') or []) for row in manual_profile['rows']),
    'graph_molecules_total': len(manual_profile['graph_payload'].get('molecules') or []),
    'graph_source_properties_total': len(manual_profile['graph_payload'].get('molecule_source_properties') or []),
}
pprint(manual_summary)
print('
Stage timings:')
for name, metrics in summarize_timings(manual_profile['stage_seconds'], molecules_kept=manual_summary['rows_kept'], entries_seen=manual_summary['entries_seen']):
    print(name, metrics)


## Variant: same chemistry, no selected-fragment file round-trip

This variant keeps `analyze_ligand_fragments(...)` unchanged, but avoids the final `current_path.write_text(...)` + `Chem.MolFromMolFile(...)` cycle for the selected fragment.
It reconstructs the selected sanitized fragment in memory from the original RDKit molecule and uses that object directly for filter evaluation.


In [ ]:
# Variant profiling: avoid selected fragment copy + reload from file

def _selected_sanitized_fragment_in_memory(mol, selected_fragment_index: int):
    fragments = Chem.GetMolFrags(Chem.Mol(mol), asMols=True, sanitizeFrags=False)
    for index, fragment in enumerate(fragments, start=1):
        if index != int(selected_fragment_index):
            continue
        try:
            return _sanitize_fragment(fragment)
        except Exception:
            return None
    return None


def profile_manual_ligand_pipeline_no_selected_roundtrip(entries_subset: list[dict]) -> dict:
    stage_seconds: Counter[str] = Counter()
    counters: Counter[str] = Counter()
    rows: list[dict] = []

    pipeline_started = time.perf_counter()
    for entry in entries_subset:
        counters['entries_seen'] += 1
        mol_block = str(entry.get('mol_block') or '')
        source_index = int(entry.get('source_index') or 0)
        name = str(entry.get('name') or f'{INPUT_FILE.stem}_{source_index}')

        t0 = time.perf_counter()
        mol = Chem.MolFromMolBlock(mol_block, sanitize=True, removeHs=False)
        stage_seconds['mol_from_mol_block'] += time.perf_counter() - t0
        if mol is None:
            counters['mol_none'] += 1
            continue

        t0 = time.perf_counter()
        paths = managed_paths_for_source(
            storage_root=storage_dir,
            role=PRIMARY_ROLE,
            source_file=INPUT_FILE,
            source_index=source_index,
            original_suffix='.sdf',
            current_suffix='.sdf',
        )
        stage_seconds['managed_paths_for_source'] += time.perf_counter() - t0
        stored_path = paths['original_path']
        current_path = paths['current_path']

        t0 = time.perf_counter()
        stored_path.write_text(mol_block, encoding='utf-8')
        stage_seconds['write_original_mol_block'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        metadata = molecule_state_metadata(mol)
        stage_seconds['molecule_state_metadata'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        row = _build_row(
            project_root=project_root,
            source_file=INPUT_FILE,
            source_index=source_index,
            name=name,
            n_atoms=mol.GetNumAtoms(),
            input_format='sdf',
            stored_path=stored_path,
            current_path=current_path,
            metadata=metadata,
            molecule_kind=MOLECULE_KIND,
            primary_role=PRIMARY_ROLE,
            primary_context=PRIMARY_CONTEXT,
        )
        stage_seconds['_build_row'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        row['source_properties'] = _source_properties_from_mapping(entry.get('source_properties'))
        stage_seconds['_source_properties_from_mapping'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        fragment_info = analyze_ligand_fragments(
            mol=mol,
            storage_root=storage_dir,
            role=PRIMARY_ROLE,
            storage_key=str(paths['key']),
            project_root=project_root,
        )
        stage_seconds['analyze_ligand_fragments'] += time.perf_counter() - t0
        if fragment_info is None:
            counters['fragment_info_none'] += 1
            continue

        t0 = time.perf_counter()
        metadata_map = dict(row.get('extra_data') or {})
        metadata_map['fragmentation'] = fragment_info
        selected_fragment = selected_fragment_from_metadata(metadata_map)
        stage_seconds['selected_fragment_from_metadata'] += time.perf_counter() - t0
        if selected_fragment is None:
            counters['selected_fragment_none'] += 1
            continue

        properties = dict(selected_fragment.get('properties') or {})
        state = dict(selected_fragment.get('state') or {})
        row['extra_data'] = metadata_map
        row['current_path'] = str(current_path.relative_to(project_root))
        row['n_atoms'] = int(properties.get('n_atoms') or row.get('n_atoms') or 0)
        for key in (
            'mw', 'exact_mw', 'logp', 'hbd', 'hba', 'tpsa', 'rotatable_bonds',
            'fragment_count', 'ring_count', 'aromatic_ring_count', 'hetero_atom_count',
            'heavy_atom_count', 'formal_charge', 'fraction_csp3',
        ):
            row[key] = properties.get(key)
        row['pains_matches'] = list(properties.get('pains_matches') or [])
        row['ro5_violations'] = list(properties.get('ro5_violations') or [])
        row['has_3d'] = bool(state.get('has_3d', row.get('has_3d')))
        row['has_hs'] = bool(state.get('has_hs', row.get('has_hs')))
        row['conformer_count'] = int(state.get('conformer_count', row.get('conformer_count') or 0) or 0)
        row['metadata_json'] = json.dumps(metadata_map, ensure_ascii=True)

        t0 = time.perf_counter()
        selected_mol = _selected_sanitized_fragment_in_memory(mol, int(selected_fragment.get('fragment_index') or 0))
        stage_seconds['selected_fragment_in_memory'] += time.perf_counter() - t0
        if selected_mol is None:
            counters['selected_mol_none'] += 1
            continue

        t0 = time.perf_counter()
        selected_report = _compute_small_molecule_filter_values(selected_mol, molecule_kind=MOLECULE_KIND)
        stage_seconds['_compute_small_molecule_filter_values'] += time.perf_counter() - t0
        if selected_report is None:
            counters['selected_report_none'] += 1
            continue

        rows.append(row)
        counters['rows_kept'] += 1

    stage_seconds['manual_no_roundtrip_total_wall'] += time.perf_counter() - pipeline_started
    graph_t0 = time.perf_counter()
    graph_payload = build_import_graph_payload(rows)
    stage_seconds['build_import_graph_payload'] += time.perf_counter() - graph_t0

    return {
        'rows': rows,
        'graph_payload': graph_payload,
        'stage_seconds': dict(stage_seconds),
        'counters': dict(counters),
    }

manual_no_roundtrip = profile_manual_ligand_pipeline_no_selected_roundtrip(entries)
manual_no_roundtrip_summary = {
    'entries_seen': manual_no_roundtrip['counters'].get('entries_seen', 0),
    'rows_kept': manual_no_roundtrip['counters'].get('rows_kept', 0),
    'rows_dropped': manual_no_roundtrip['counters'].get('entries_seen', 0) - manual_no_roundtrip['counters'].get('rows_kept', 0),
    'graph_molecules_total': len(manual_no_roundtrip['graph_payload'].get('molecules') or []),
    'graph_source_properties_total': len(manual_no_roundtrip['graph_payload'].get('molecule_source_properties') or []),
}
pprint(manual_no_roundtrip_summary)
print('
Stage timings:')
for name, metrics in summarize_timings(manual_no_roundtrip['stage_seconds'], molecules_kept=manual_no_roundtrip_summary['rows_kept'], entries_seen=manual_no_roundtrip_summary['entries_seen']):
    print(name, metrics)


In [ ]:
# Full helper profiling on the same subset
batch_payload = make_batch_payload(entries)

helper_t0 = time.perf_counter()
helper_rows = materialize_import_batch(batch_payload)
helper_t1 = time.perf_counter()
helper_graph = build_import_graph_payload(helper_rows)
helper_t2 = time.perf_counter()

helper_summary = {
    'rows_total': len(helper_rows),
    'rows_dropped': len(entries) - len(helper_rows),
    'graph_molecules_total': len(helper_graph.get('molecules') or []),
    'graph_source_properties_total': len(helper_graph.get('molecule_source_properties') or []),
    'materialize_import_batch_s': round(helper_t1 - helper_t0, 6),
    'build_import_graph_payload_s': round(helper_t2 - helper_t1, 6),
    'helper_total_wall_s': round(helper_t2 - helper_t0, 6),
}
pprint(helper_summary)


## Variant: in-memory fragment analysis without fragment artifact writes

This variant removes the largest file-producing step inside ligand handling: fragment artifact generation in `analyze_ligand_fragments(...)`.
It keeps the fragment split, sanitation, descriptor calculation, selection strategy, and final filter evaluation, but builds fragment metadata in memory only.


In [ ]:
# Variant profiling: no fragment artifact writes

def analyze_ligand_fragments_in_memory_no_artifacts(*, mol):
    fragments = Chem.GetMolFrags(Chem.Mol(mol), asMols=True, sanitizeFrags=False)
    if not fragments:
        return None

    component_rows = []
    problem_flags = []
    for index, fragment in enumerate(fragments, start=1):
        try:
            sanitized = _sanitize_fragment(fragment)
        except Exception:
            problem_flags.append(f'unsanitizable_fragment_{index}')
            continue
        values = evaluate_small_molecule_filter_values(sanitized)
        component_rows.append(
            {
                'fragment_index': int(index),
                'role': _component_role(sanitized),
                'path': '',
                'smiles': _canonical_smiles(sanitized),
                'formula': _formula(sanitized),
                'atom_count': int(sanitized.GetNumAtoms() or 0),
                'heavy_atom_count': int(sanitized.GetNumHeavyAtoms() or 0),
                'formal_charge': int(sum(atom.GetFormalCharge() for atom in sanitized.GetAtoms())),
                'contains_metal': bool(_has_metal_atoms(sanitized)),
                'is_simple_ion': bool(_is_simple_ion(sanitized)),
                'has_carbon': bool(_has_carbon(sanitized)),
                'state': dict(molecule_state_metadata(sanitized).get('state') or {}),
                'properties': {
                    'n_atoms': int(sanitized.GetNumAtoms() or 0),
                    'mw': values.descriptors.get('mw'),
                    'exact_mw': values.descriptors.get('exact_mw'),
                    'logp': values.descriptors.get('logp'),
                    'hbd': values.descriptors.get('hbd'),
                    'hba': values.descriptors.get('hba'),
                    'tpsa': values.descriptors.get('tpsa'),
                    'rotatable_bonds': values.descriptors.get('rotatable_bonds'),
                    'fragment_count': values.descriptors.get('fragment_count'),
                    'ring_count': values.descriptors.get('ring_count'),
                    'aromatic_ring_count': values.descriptors.get('aromatic_ring_count'),
                    'hetero_atom_count': values.descriptors.get('hetero_atom_count'),
                    'heavy_atom_count': values.descriptors.get('heavy_atom_count'),
                    'formal_charge': values.descriptors.get('formal_charge'),
                    'fraction_csp3': values.descriptors.get('fraction_csp3'),
                    'pains_matches': list(values.pains_matches),
                    'ro5_violations': list(values.ro5_violations),
                },
            }
        )

    if not component_rows:
        return None

    candidate_rows = [row for row in component_rows if row.get('role') == 'fragment']
    organic_candidates = [row for row in candidate_rows if bool(row.get('has_carbon'))]
    ranked = organic_candidates or candidate_rows
    if not ranked:
        return {
            'selected_fragment_index': None,
            'selection_strategy': 'none',
            'problem_flags': sorted(set(problem_flags + ['no_valid_ligand_fragment'])),
            'components': component_rows,
        }

    selected = sorted(
        ranked,
        key=lambda row: (
            int(row.get('heavy_atom_count') or 0),
            int(row.get('atom_count') or 0),
            str(row.get('smiles') or ''),
        ),
        reverse=True,
    )[0]
    selected_index = int(selected.get('fragment_index') or 0)
    for row in component_rows:
        if int(row.get('fragment_index') or 0) == selected_index:
            row['role'] = 'selected_fragment'
            break

    detached_protons = sum(1 for row in component_rows if str(row.get('role') or '') == 'detached_proton')
    associated_ions = sum(1 for row in component_rows if str(row.get('role') or '') == 'associated_ion')
    associated_components = sum(1 for row in component_rows if str(row.get('role') or '') == 'associated_component')
    unique_nonselected = {
        str(row.get('smiles') or '')
        for row in component_rows
        if int(row.get('fragment_index') or 0) != selected_index and str(row.get('smiles') or '').strip()
    }
    if detached_protons:
        problem_flags.append('detached_protons')
    if associated_ions:
        problem_flags.append('associated_ions')
    if associated_components:
        problem_flags.append('associated_components')
    if len(unique_nonselected) > 1:
        problem_flags.append('multiple_distinct_nonselected_components')

    return {
        'selected_fragment_index': selected_index,
        'selection_strategy': 'largest_organic_fragment' if organic_candidates else 'largest_non_ion_fragment',
        'problem_flags': sorted(set(problem_flags)),
        'components': component_rows,
        'detached_proton_count': detached_protons,
        'associated_ion_count': associated_ions,
        'associated_component_count': associated_components,
    }


def profile_manual_ligand_pipeline_no_fragment_artifacts(entries_subset: list[dict]) -> dict:
    stage_seconds: Counter[str] = Counter()
    counters: Counter[str] = Counter()
    rows: list[dict] = []

    pipeline_started = time.perf_counter()
    for entry in entries_subset:
        counters['entries_seen'] += 1
        mol_block = str(entry.get('mol_block') or '')
        source_index = int(entry.get('source_index') or 0)
        name = str(entry.get('name') or f'{INPUT_FILE.stem}_{source_index}')

        t0 = time.perf_counter()
        mol = Chem.MolFromMolBlock(mol_block, sanitize=True, removeHs=False)
        stage_seconds['mol_from_mol_block'] += time.perf_counter() - t0
        if mol is None:
            counters['mol_none'] += 1
            continue

        t0 = time.perf_counter()
        paths = managed_paths_for_source(
            storage_root=storage_dir,
            role=PRIMARY_ROLE,
            source_file=INPUT_FILE,
            source_index=source_index,
            original_suffix='.sdf',
            current_suffix='.sdf',
        )
        stage_seconds['managed_paths_for_source'] += time.perf_counter() - t0
        stored_path = paths['original_path']
        current_path = paths['current_path']

        t0 = time.perf_counter()
        stored_path.write_text(mol_block, encoding='utf-8')
        stage_seconds['write_original_mol_block'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        metadata = molecule_state_metadata(mol)
        stage_seconds['molecule_state_metadata'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        row = _build_row(
            project_root=project_root,
            source_file=INPUT_FILE,
            source_index=source_index,
            name=name,
            n_atoms=mol.GetNumAtoms(),
            input_format='sdf',
            stored_path=stored_path,
            current_path=current_path,
            metadata=metadata,
            molecule_kind=MOLECULE_KIND,
            primary_role=PRIMARY_ROLE,
            primary_context=PRIMARY_CONTEXT,
        )
        stage_seconds['_build_row'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        row['source_properties'] = _source_properties_from_mapping(entry.get('source_properties'))
        stage_seconds['_source_properties_from_mapping'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        fragment_info = analyze_ligand_fragments_in_memory_no_artifacts(mol=mol)
        stage_seconds['analyze_ligand_fragments_in_memory_no_artifacts'] += time.perf_counter() - t0
        if fragment_info is None:
            counters['fragment_info_none'] += 1
            continue

        t0 = time.perf_counter()
        metadata_map = dict(row.get('extra_data') or {})
        metadata_map['fragmentation'] = fragment_info
        selected_fragment = selected_fragment_from_metadata(metadata_map)
        stage_seconds['selected_fragment_from_metadata'] += time.perf_counter() - t0
        if selected_fragment is None:
            counters['selected_fragment_none'] += 1
            continue

        properties = dict(selected_fragment.get('properties') or {})
        state = dict(selected_fragment.get('state') or {})
        row['extra_data'] = metadata_map
        row['current_path'] = str(current_path.relative_to(project_root))
        row['n_atoms'] = int(properties.get('n_atoms') or row.get('n_atoms') or 0)
        for key in (
            'mw', 'exact_mw', 'logp', 'hbd', 'hba', 'tpsa', 'rotatable_bonds',
            'fragment_count', 'ring_count', 'aromatic_ring_count', 'hetero_atom_count',
            'heavy_atom_count', 'formal_charge', 'fraction_csp3',
        ):
            row[key] = properties.get(key)
        row['pains_matches'] = list(properties.get('pains_matches') or [])
        row['ro5_violations'] = list(properties.get('ro5_violations') or [])
        row['has_3d'] = bool(state.get('has_3d', row.get('has_3d')))
        row['has_hs'] = bool(state.get('has_hs', row.get('has_hs')))
        row['conformer_count'] = int(state.get('conformer_count', row.get('conformer_count') or 0) or 0)
        row['metadata_json'] = json.dumps(metadata_map, ensure_ascii=True)

        t0 = time.perf_counter()
        selected_mol = _selected_sanitized_fragment_in_memory(mol, int(selected_fragment.get('fragment_index') or 0))
        stage_seconds['selected_fragment_in_memory'] += time.perf_counter() - t0
        if selected_mol is None:
            counters['selected_mol_none'] += 1
            continue

        t0 = time.perf_counter()
        selected_report = _compute_small_molecule_filter_values(selected_mol, molecule_kind=MOLECULE_KIND)
        stage_seconds['_compute_small_molecule_filter_values'] += time.perf_counter() - t0
        if selected_report is None:
            counters['selected_report_none'] += 1
            continue

        rows.append(row)
        counters['rows_kept'] += 1

    stage_seconds['manual_no_fragment_artifacts_total_wall'] += time.perf_counter() - pipeline_started
    graph_t0 = time.perf_counter()
    graph_payload = build_import_graph_payload(rows)
    stage_seconds['build_import_graph_payload'] += time.perf_counter() - graph_t0

    return {
        'rows': rows,
        'graph_payload': graph_payload,
        'stage_seconds': dict(stage_seconds),
        'counters': dict(counters),
    }

manual_no_fragment_artifacts = profile_manual_ligand_pipeline_no_fragment_artifacts(entries)
manual_no_fragment_artifacts_summary = {
    'entries_seen': manual_no_fragment_artifacts['counters'].get('entries_seen', 0),
    'rows_kept': manual_no_fragment_artifacts['counters'].get('rows_kept', 0),
    'rows_dropped': manual_no_fragment_artifacts['counters'].get('entries_seen', 0) - manual_no_fragment_artifacts['counters'].get('rows_kept', 0),
    'graph_molecules_total': len(manual_no_fragment_artifacts['graph_payload'].get('molecules') or []),
    'graph_source_properties_total': len(manual_no_fragment_artifacts['graph_payload'].get('molecule_source_properties') or []),
}
pprint(manual_no_fragment_artifacts_summary)
print('
Stage timings:')
for name, metrics in summarize_timings(manual_no_fragment_artifacts['stage_seconds'], molecules_kept=manual_no_fragment_artifacts_summary['rows_kept'], entries_seen=manual_no_fragment_artifacts_summary['entries_seen']):
    print(name, metrics)


In [ ]:
# Comparison table
comparison = {
    'manual_total_wall_s': round(manual_profile['stage_seconds'].get('manual_total_wall', 0.0), 6),
    'manual_no_roundtrip_total_wall_s': round(manual_no_roundtrip['stage_seconds'].get('manual_no_roundtrip_total_wall', 0.0), 6),
    'manual_no_fragment_artifacts_total_wall_s': round(manual_no_fragment_artifacts['stage_seconds'].get('manual_no_fragment_artifacts_total_wall', 0.0), 6),
    'helper_total_wall_s': helper_summary['helper_total_wall_s'],
    'same_rows_manual_vs_no_roundtrip': len(manual_profile['rows']) == len(manual_no_roundtrip['rows']),
    'same_rows_manual_vs_no_fragment_artifacts': len(manual_profile['rows']) == len(manual_no_fragment_artifacts['rows']),
    'same_rows_manual_vs_helper': len(manual_profile['rows']) == len(helper_rows),
    'same_graph_molecules_manual_vs_no_roundtrip': manual_summary['graph_molecules_total'] == manual_no_roundtrip_summary['graph_molecules_total'],
    'same_graph_molecules_manual_vs_no_fragment_artifacts': manual_summary['graph_molecules_total'] == manual_no_fragment_artifacts_summary['graph_molecules_total'],
    'same_graph_molecules_manual_vs_helper': manual_summary['graph_molecules_total'] == helper_summary['graph_molecules_total'],
    'saved_roundtrip_only_s': round(manual_profile['stage_seconds'].get('manual_total_wall', 0.0) - manual_no_roundtrip['stage_seconds'].get('manual_no_roundtrip_total_wall', 0.0), 6),
    'saved_roundtrip_only_pct': round(((manual_profile['stage_seconds'].get('manual_total_wall', 0.0) - manual_no_roundtrip['stage_seconds'].get('manual_no_roundtrip_total_wall', 0.0)) / max(1e-9, manual_profile['stage_seconds'].get('manual_total_wall', 0.0))) * 100.0, 2),
    'saved_fragment_artifacts_s': round(manual_profile['stage_seconds'].get('manual_total_wall', 0.0) - manual_no_fragment_artifacts['stage_seconds'].get('manual_no_fragment_artifacts_total_wall', 0.0), 6),
    'saved_fragment_artifacts_pct': round(((manual_profile['stage_seconds'].get('manual_total_wall', 0.0) - manual_no_fragment_artifacts['stage_seconds'].get('manual_no_fragment_artifacts_total_wall', 0.0)) / max(1e-9, manual_profile['stage_seconds'].get('manual_total_wall', 0.0))) * 100.0, 2),
    'manual_molecules_per_s': round(manual_summary['rows_kept'] / max(1e-9, manual_profile['stage_seconds'].get('manual_total_wall', 0.0)), 3),
    'manual_no_roundtrip_molecules_per_s': round(manual_no_roundtrip_summary['rows_kept'] / max(1e-9, manual_no_roundtrip['stage_seconds'].get('manual_no_roundtrip_total_wall', 0.0)), 3),
    'manual_no_fragment_artifacts_molecules_per_s': round(manual_no_fragment_artifacts_summary['rows_kept'] / max(1e-9, manual_no_fragment_artifacts['stage_seconds'].get('manual_no_fragment_artifacts_total_wall', 0.0)), 3),
    'helper_molecules_per_s': round(helper_summary['rows_total'] / max(1e-9, helper_summary['helper_total_wall_s']), 3),
}
pprint(comparison)


## Notes and next steps

- Focus first on the largest stages inside the manual loop.
- If `analyze_ligand_fragments` dominates, the main issue is chemistry plus artifact generation, not scheduler overhead.
- The `no_roundtrip` variant isolates only the final selected-fragment disk handoff.
- The `no_fragment_artifacts` variant removes the heavier disk-producing part inside fragment analysis itself.
- If `no_fragment_artifacts` is materially faster and preserves row counts, then fragment artifact generation is a first-class optimization target.
- If even `no_fragment_artifacts` is only slightly faster, then the dominant cost is RDKit chemistry and descriptor/filter computation itself.
